# Apache Hudi

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/hudi/hudi.ipynb)

Apache Hudi (Hadoop Upserts Deletes and Incrementals) is an open table format and lakehouse framework for record-level upserts, deletes, and incremental pulls on data lakes. Hudi has no lightweight Python engine, so Spark snippets are shown as fenced code and the runnable cells use a small pure-Python model of Hudi's timeline, record keys, file groups, and COW/MOR mechanics.

## What is Apache Hudi?

Apache Hudi is an open-source data lakehouse framework that brings database-like capabilities (upserts, deletes, incremental pulls) to data lakes on object storage. It manages table metadata and timelines so batch and streaming engines can write and read consistently. Hudi is not a query engine: Spark, Flink, Trino, Presto, Hive, and BigQuery-style external readers use Hudi metadata to read the same table.

+ A **timeline** in `.hoodie/` records every instant (commit, deltacommit, compaction, clean, replacecommit) and lets readers see a consistent snapshot.
+ A **record key index** lets an upsert locate the file group holding a key instead of rewriting the whole table.
+ **Table services** (compaction, clustering, cleaning, archival) run inline or asynchronously to keep the table healthy.
+ **Incremental queries** return only records changed since an instant, which enables streaming-on-batch pipelines.

```text
Spark / Flink / DeltaStreamer
        |
        v
s3://lake/hudi/orders/
  .hoodie/                 table metadata + timeline
    hoodie.properties
    metadata/              metadata table (files + column stats index)
    20261004013000.commit    completed instant
    20261004020000.deltacommit
  order_date=2026-10-04/   partitions
    <fileId>_<token>_<instant>.parquet     base files
```

A first write and read in PySpark (needs the Hudi Spark bundle):

```python
hudi_options = {
    "hoodie.table.name": "orders_hudi",
    "hoodie.datasource.write.recordkey.field": "order_id",
    "hoodie.datasource.write.partitionpath.field": "order_date",
    "hoodie.datasource.write.precombine.field": "updated_at",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.operation": "upsert",
}
orders_df.write.format("hudi").options(**hudi_options).mode("append").save("s3://lake/hudi/orders")
orders = spark.read.format("hudi").load("s3://lake/hudi/orders")
```

How a Hudi table is organized under the hood. Every write creates an instant on the timeline. An instant moves through `requested`, `inflight`, and `completed`. Readers only see completed instants, which is what gives snapshot isolation. Data lives in file groups. A file group is one `fileId` in one partition. In a COW table a file group is a stack of base file versions. In a MOR table it is a base file plus one or more log files. The set of files a reader needs at one instant is a file slice: the latest base file plus the log files newer than it.

The model below implements those pieces: a timeline of instants, a record-key index pointing at file groups, base files and (for MOR) log files, precombine-based deduplication, and the three query types. Later cells reuse it.

In [ ]:
import itertools
import zlib
from dataclasses import dataclass, field


@dataclass
class FileGroup:
    file_id: str
    partition: str
    base_versions: list = field(default_factory=list)  # [(instant, {key: row})], newest last
    log_files: list = field(default_factory=list)      # [(instant, {key: row or None})] since the latest base

    def latest_slice(self) -> dict:
        """Merge the latest base file with newer log files (a MOR snapshot read)."""
        rows = dict(self.base_versions[-1][1]) if self.base_versions else {}
        for _, delta in self.log_files:
            for key, row in delta.items():
                if row is None:
                    rows.pop(key, None)
                else:
                    rows[key] = row
        return rows


class HudiTable:
    """A tiny in-memory model of a Hudi table: timeline, index, file groups."""

    def __init__(self, table_type: str, record_key="order_id", partition_path="order_date",
                 precombine="updated_at", max_records_per_file=3):
        assert table_type in ("COPY_ON_WRITE", "MERGE_ON_READ")
        self.table_type = table_type
        self.record_key, self.partition_path, self.precombine = record_key, partition_path, precombine
        self.max_records_per_file = max_records_per_file
        self.timeline: list[dict] = []
        self.index: dict = {}            # record key -> file_id (a global map, like an HBase/record index)
        self.file_groups: dict[str, FileGroup] = {}
        self._clock = itertools.count(1)

    # -- timeline ---------------------------------------------------------
    def _begin(self, action: str) -> str:
        instant = f"20261004{next(self._clock):06d}"
        self.timeline.append({"instant": instant, "action": action, "state": "REQUESTED"})
        self.timeline[-1]["state"] = "INFLIGHT"
        return instant

    def _complete(self, instant: str) -> None:
        next(e for e in self.timeline if e["instant"] == instant)["state"] = "COMPLETED"

    def completed(self) -> list[str]:
        return [e["instant"] for e in self.timeline if e["state"] == "COMPLETED"]

    # -- writes -----------------------------------------------------------
    def _dedupe(self, records):
        latest = {}
        for row in records:  # precombine: keep the greatest ordering value per key within the batch
            key = row[self.record_key]
            if key not in latest or row[self.precombine] >= latest[key][self.precombine]:
                latest[key] = row
        return latest

    def _write(self, changes: dict, instant: str) -> None:
        by_group: dict[str, dict] = {}
        for key, row in changes.items():
            file_id = self.index.get(key)
            if file_id is None:
                if row is None:
                    continue  # delete of an unknown key: nothing to do
                file_id = self._file_for_insert(row[self.partition_path], by_group)
                self.index[key] = file_id
            by_group.setdefault(file_id, {})[key] = row
        for file_id, delta in by_group.items():
            group = self.file_groups[file_id]
            if self.table_type == "MERGE_ON_READ" and group.base_versions:
                group.log_files.append((instant, delta))              # append a delta log, no rewrite
            else:
                merged = group.latest_slice()                         # copy-on-write: rewrite the file
                for key, row in delta.items():
                    if row is None:
                        merged.pop(key, None)
                    else:
                        merged[key] = row
                group.base_versions.append((instant, merged))
                group.log_files.clear()
        for key, row in changes.items():
            if row is None:
                self.index.pop(key, None)

    def _file_for_insert(self, partition: str, pending: dict) -> str:
        for group in self.file_groups.values():  # small-file handling: fill an under-sized group first
            size = len(group.latest_slice()) + len(pending.get(group.file_id, {}))
            if group.partition == partition and size < self.max_records_per_file:
                return group.file_id
        file_id = f"fg{len(self.file_groups) + 1}"
        self.file_groups[file_id] = FileGroup(file_id, partition)
        return file_id

    def upsert(self, records) -> str:
        instant = self._begin("deltacommit" if self.table_type == "MERGE_ON_READ" else "commit")
        changes = {}
        current = self.snapshot_by_key()
        for key, row in self._dedupe(records).items():
            stored = current.get(key)
            if stored is not None and row[self.precombine] < stored[self.precombine]:
                continue  # late-arriving older version: ignored (DefaultHoodieRecordPayload semantics)
            changes[key] = {**row, "_hoodie_commit_time": instant, "_hoodie_record_key": key}
        self._write(changes, instant)
        self._complete(instant)
        return instant

    def delete(self, keys) -> str:
        instant = self._begin("deltacommit" if self.table_type == "MERGE_ON_READ" else "commit")
        self._write({key: None for key in keys}, instant)
        self._complete(instant)
        return instant

    # -- table services ---------------------------------------------------
    def compact(self) -> str:
        instant = self._begin("compaction")
        for group in self.file_groups.values():
            if group.log_files:
                group.base_versions.append((instant, group.latest_slice()))
                group.log_files.clear()
        self._complete(instant)
        return instant

    def clean(self, versions_retained: int) -> int:
        removed = 0
        for group in self.file_groups.values():
            extra = len(group.base_versions) - versions_retained
            if extra > 0:
                del group.base_versions[:extra]
                removed += extra
        return removed

    # -- queries ----------------------------------------------------------
    def snapshot_by_key(self) -> dict:
        rows = {}
        for group in self.file_groups.values():
            rows.update(group.latest_slice())
        return rows

    def snapshot(self) -> list[dict]:
        return sorted(self.snapshot_by_key().values(), key=lambda r: r[self.record_key])

    def read_optimized(self) -> list[dict]:
        rows = {}
        for group in self.file_groups.values():
            if group.base_versions:
                rows.update(group.base_versions[-1][1])
        return sorted(rows.values(), key=lambda r: r[self.record_key])

    def incremental(self, begin: str, end: str | None = None) -> list[dict]:
        return [r for r in self.snapshot() if r["_hoodie_commit_time"] > begin
                and (end is None or r["_hoodie_commit_time"] <= end)]


def order(order_id, status, updated_at, date="2026-10-04", amount=10.0):
    return {"order_id": order_id, "order_date": date, "status": status, "updated_at": updated_at, "amount": amount}


print("model ready")

## What problems does Hudi solve in a data lake?

+ **Updating a small fraction of a large table.** Rewriting 50 TB of Parquet to change 1% of rows is wasteful. Hudi rewrites only the affected file groups.
+ **Deletes and GDPR erasure.** Path-based lakes cannot delete a single customer's rows cheaply. Hudi supports record-level deletes and a delete-marker payload for soft deletes.
+ **Incremental processing.** Extracting "what changed since the last run" from Parquet means scanning and diffing. Hudi's incremental query returns exactly the changed records since an instant.
+ **Small files and write amplification.** Frequent micro-batch ingestion creates thousands of tiny files. Hudi's file-sizing, cleaning, and clustering services bound file counts and file sizes.
+ **Consistency for concurrent readers and writers.** A commit either fully succeeds or is rolled back; readers see the last successful instant even while writers are appending.

## What are Hudi's table types and physical file layout?

Hudi has two table types, `COPY_ON_WRITE` (COW) and `MERGE_ON_READ` (MOR), and a file-group abstraction shared by both.

+ A **file group** is identified by a stable `fileId` and lives in one partition. All versions of a record's data land in the same file group, which is what makes the index lookup cheap.
+ A **base file** is a columnar file, usually Parquet, named `<fileId>_<writeToken>_<instantTime>.parquet`.
+ A **log file** (MOR only) is an Avro-encoded delta log named `.<fileId>_<baseInstantTime>.log.<version>_<writeToken>`; it holds changes until compaction folds them into a new base file.
+ The **`.hoodie` directory** holds `hoodie.properties` (table config), the active timeline, archived timeline, and the optional metadata table.

```text
s3://lake/hudi/orders/
  .hoodie/
    hoodie.properties
    20261004013000.commit
    20261004020000.commit.requested
    metadata/
      files/            file listing index
      column_stats/     column statistics index
  order_date=2026-10-04/
    7f3a2c1b-...-0_20261004013000.parquet
    .7f3a2c1b-...-0_20261004013000.log.1_20261004020000
```

| Action | Meaning |
|---|---|
| `commit` | A completed COW write |
| `deltacommit` | A completed MOR write that appended log files |
| `replacecommit` | Atomically replaces file groups; used by clustering and insert-overwrite |
| `compaction` | Folds MOR log files into new base files |
| `clean` | Deletes old file versions and log files beyond retention |
| `rollback` | Removes a failed or aborted instant's files |
| `savepoint` | Pins a file version so cleaning cannot remove it |

Every Hudi record carries hidden metadata columns: `_hoodie_commit_time`, `_hoodie_commit_seqno`, `_hoodie_record_key`, `_hoodie_partition_path`, and `_hoodie_file_name`. They are how Hudi tracks record provenance per row.

Timeline states. An instant starts as `<instant>.requested`, becomes `<instant>.inflight` while a writer holds it, and finishes as `<instant>.<action>` when complete. The action is one of commit, deltacommit, replacecommit, compaction, clean, rollback, or savepoint.

Three writes to a COW table, then the timeline and the file-group versions:

In [ ]:
cow = HudiTable("COPY_ON_WRITE")
c1 = cow.upsert([order(1001, "CREATED", 1), order(1002, "CREATED", 1), order(1003, "CREATED", 1)])
c2 = cow.upsert([order(1002, "PAID", 2), order(1004, "CREATED", 2)])
c3 = cow.upsert([order(1001, "SHIPPED", 3)])

for entry in cow.timeline:
    print(entry)
for group in cow.file_groups.values():
    print(group.file_id, group.partition, "base versions at", [instant for instant, _ in group.base_versions])

assert [e["action"] for e in cow.timeline] == ["commit"] * 3
assert all(e["state"] == "COMPLETED" for e in cow.timeline)
assert len(cow.file_groups["fg1"].base_versions) == 3  # every touch rewrote fg1 (copy-on-write)

## How does Hudi handle upserts and deletes?

Hudi manages record-level changes on a data lake table using a record key, precombine field, and table timeline. It can apply inserts, updates, and deletes efficiently compared with rewriting an entire dataset.

1. Hudi reads the input records, computes each record key, and resolves duplicates using the precombine field.
2. The index maps each key to an existing file group, or tags the record as an insert for a new file group.
3. Hudi groups records by file group, reads only those base files (plus log files on MOR), merges the new records, and writes the result.
4. A new instant is committed to the timeline; readers atomically switch to the new snapshot.

+ **Hard delete**: the record is physically removed via the `delete` operation, which targets the file groups holding the keys.
+ **Soft delete**: the record is written with a null ordering value (or a delete marker), so readers filter it out while the row remains for audit. Hudi's `DefaultHoodieRecordPayload` treats a record whose precombine value is null as a delete.

Precombine in action: duplicates inside a batch keep the greatest `updated_at`, and a late event older than the stored row is ignored. Then a hard delete:

In [ ]:
cow.upsert([order(1003, "PAID", 5), order(1003, "CANCELLED", 4)])   # duplicate keys in one batch
cow.upsert([order(1003, "CREATED", 2)])                               # late, older event
print({r["order_id"]: r["status"] for r in cow.snapshot()})
assert next(r for r in cow.snapshot() if r["order_id"] == 1003)["status"] == "PAID"

cow.delete([1004])
print("after delete:", [r["order_id"] for r in cow.snapshot()])
assert 1004 not in {r["order_id"] for r in cow.snapshot()}
assert 1004 not in cow.index

Common mistake: a soft delete that nulls a data column but keeps a non-null ordering value. The row stays live and the delete silently does nothing. Null the precombine field, or switch to a delete-marker payload so the intent is explicit.

## How does the index make upserts cheap?

Hudi indexes records by key and can locate existing records to update them. On upsert, it writes new versions and updates the timeline/metadata so readers see a consistent view. Correctness depends on stable keys and proper handling of late and duplicate events.

```text
incoming batch (1M changed orders)
    -> key generation (order_id)
    -> index lookup (bloom filter / HFile / bucket)
    -> touched file groups (2,000 of 500,000)
    -> merge + write 2,000 new base files
```

+ Choose a stable, immutable record key. If the key changes when the business entity changes, you get duplicates instead of updates.
+ Provide a precombine field that reflects event time (`updated_at`, `source_lsn`) rather than ingestion time; otherwise replays and late events win incorrectly.
+ Keep record keys evenly distributed so the shuffle and the index do not create a single hot task.
+ Control the `hoodie.upsert.shuffle.parallelism` to roughly match data volume; too few tasks cause long, memory-heavy merges.
+ Use `bulk_insert` for the initial historical load and `upsert` only for incremental changes; bulk insert skips the index and is far faster for empty tables.

## What is a record key, partition path, and precombine field?

Record key identifies a row uniquely (like a primary key). Partition path controls how data is grouped for storage and pruning. Precombine field is used to resolve multiple records for the same key in a batch (for example choose the latest by `updated_at`) to ensure deterministic upserts.

+ The partition path must be derivable from the record or supplied as a literal string. A partition column with thousands of values (for example `order_id`) creates tiny partitions and destroys file sizing.
+ Composite record keys are supported with `ComplexKeyGenerator` and a comma-separated list, for example `"tenant_id,order_id"`.
+ Timestamp-derived partitions use `TimestampBasedKeyGenerator` with `hoodie.keygen.timebased.timestamp.type` and format options.
+ The precombine comparison is per key. Ties (equal ordering values) are resolved deterministically by the payload, but you should not depend on arbitrary tie-breaking; make the ordering value unique per event where possible.
+ Non-partitioned tables use `NonpartitionedKeyGenerator` and live entirely at the table root, which is fine for small tables but loses pruning on large ones.

## How do you write and upsert data with Hudi in PySpark?

A Hudi write is a normal `DataFrameWriter` with Hudi options. The options decide table type, operation, key semantics, indexing, and table services. This is the example most interviews ask you to produce from memory.

```python
hudi_options = {
    "hoodie.table.name": "orders_hudi",
    "hoodie.datasource.write.recordkey.field": "order_id",
    "hoodie.datasource.write.partitionpath.field": "order_date",
    "hoodie.datasource.write.precombine.field": "updated_at",
    "hoodie.datasource.write.table.type": "MERGE_ON_READ",
    "hoodie.datasource.write.operation": "upsert",
    "hoodie.datasource.write.hive_style_partitioning": "true",
    "hoodie.index.type": "BLOOM",
    "hoodie.upsert.shuffle.parallelism": "200",
    "hoodie.parquet.max.file.size": "134217728",
    "hoodie.parquet.small.file.limit": "104857600",
    "hoodie.metadata.enable": "true",
    "hoodie.clean.automatic": "true",
    "hoodie.cleaner.policy": "KEEP_LATEST_COMMITS",
    "hoodie.cleaner.commits.retained": "10",
    "hoodie.compact.inline": "true",
    "hoodie.compact.inline.max.delta.commits": "5",
}
orders_df.write.format("hudi").options(**hudi_options).mode("append").save("s3://lake/hudi/orders")

# Initial historical load: bulk_insert skips the index lookup
history_df.write.format("hudi").options(**{**hudi_options, "hoodie.datasource.write.operation": "bulk_insert"}) \
    .mode("overwrite").save("s3://lake/hudi/orders")

# Replace one business day atomically (replacecommit)
day_df.write.format("hudi").options(**{**hudi_options, "hoodie.datasource.write.operation": "insert_overwrite"}) \
    .mode("append").save("s3://lake/hudi/orders")
```

+ `operation=upsert` with `mode("append")` is the normal incremental path; `mode("overwrite")` on an existing table is usually wrong and can wipe history.
+ `hoodie.datasource.write.table.type` is fixed when the table is created. Converting COW to MOR requires a rewrite or a migration job.
+ In Databricks you can register the Hudi table in Unity Catalog with `CREATE TABLE ... USING hudi LOCATION ...`, but check runtime support for the Hudi version you depend on.
+ Keep the Hudi bundle version aligned with the Spark version (`hudi-spark3.5-bundle_2.12` for Spark 3.5), because Hudi and Spark internals are tightly coupled.

```sql
-- Register an existing Hudi path as a Spark SQL table.
CREATE TABLE orders_hudi
USING hudi
LOCATION 's3://lake/hudi/orders';
```

```sql
MERGE INTO orders_hudi AS t
USING staged_orders AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.op = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

Choosing the operation:

+ `upsert` looks up every key in the index and merges. Use it for incremental changes.
+ `insert` writes new rows without merging and keeps small-file handling. Use it for an append-only feed that is known to be duplicate-free.
+ `bulk_insert` skips the index entirely and bin-packs records into files. Use it for the initial load of an empty table, never on an existing table.

## What indexes does Hudi use to find records quickly?

Hudi's index maps record keys to file groups so an upsert only touches affected files. The index choice is one of the biggest levers on upsert latency and write amplification.

| Index | How it works | Best for |
|---|---|---|
| `BLOOM` (default) | Bloom filter per base file, probed by the incoming keys | Large partitioned tables, most upsert workloads |
| `SIMPLE` | Keeps a full key-to-file map in memory | Small tables that fit in executor memory |
| `GLOBAL_BLOOM` | Bloom filter across all partitions | Global uniqueness when keys can move partitions |
| `HFILE` | Stores the key map in an HFile per file group | Large tables where a persistent, fast lookup beats recomputing blooms |
| `BUCKET` | Hash of the key determines the file group; no lookup at all | Streaming ingestion with a fixed bucket count |

+ Bloom indexes trade memory and CPU on every upsert for flexibility; bucket indexes trade flexibility for near-zero lookup cost.
+ `BLOOM` can produce false positives, which only cause an unnecessary file read, never a wrong result.
+ Global indexes cost more because the key space spans partitions; use local indexes unless keys genuinely move between partitions.
+ The metadata table (`hoodie.metadata.enable=true`) stores the files index and column statistics index, which speeds up query planning by avoiding repeated object-store listing. It is effectively mandatory on cloud object storage.

How the lookup works. A Bloom index keeps a small filter per base file. Hudi probes the filters with the incoming keys and reads only the files whose filter may match. A false positive causes one extra file read, never a wrong answer. An HFile index stores the key-to-fileId mapping in a sorted, immutable HFile per file group, which avoids rebuilding filters on every upsert. A simple index keeps the whole key map in memory on the driver, which is fast and fragile once the table grows. A bucket index hashes the key to a bucket, so there is no lookup at all.

A bucket index replaces the lookup with a hash: `bucket = hash(record_key) % num_buckets`. Any writer computes the file group without reading anything, which is why it suits streaming, and why the bucket count is fixed at creation:

In [ ]:
NUM_BUCKETS = 8


def bucket_for(record_key: str) -> int:
    return zlib.crc32(record_key.encode()) % NUM_BUCKETS


placements = {key: bucket_for(key) for key in ["O-1001", "O-1002", "O-1003", "O-1001"]}
print(placements)
assert placements["O-1001"] == bucket_for("O-1001")  # deterministic: the same key always lands in the same bucket
assert len({bucket_for(f"O-{i}") for i in range(1_000)}) == NUM_BUCKETS

## What is the difference between Copy-on-Write (COW) and Merge-on-Read (MOR)?

COW stores data in columnar files and rewrites them on updates, optimizing read performance at the cost of write amplification. MOR stores base files plus delta log files and merges them at read time (or during compaction), optimizing write performance at the cost of more complex reads.

| Dimension | COW | MOR |
|---|---|---|
| Upsert write cost | Rewrites affected base files | Appends small log files |
| Read cost (snapshot) | Low, pure columnar | Higher, merges base + logs |
| Read cost (read_optimized) | Same as snapshot | Ignores logs, may be stale |
| Storage overhead | Multiple file versions until cleaning | Versions plus log files |
| Compaction requirement | None | Essential; backlog degrades reads |
| Streaming ingestion | Higher latency | Low latency |
| Engine compatibility | Broadest | Requires Hudi-aware readers for logs |
| Typical partition size | Moderate to large | Large, write-heavy |

The same writes on a MOR table append log files instead of rewriting base files:

In [ ]:
batches = [
    [order(1001, "CREATED", 1), order(1002, "CREATED", 1), order(1003, "CREATED", 1)],
    [order(1002, "PAID", 2), order(1004, "CREATED", 2)],
    [order(1001, "SHIPPED", 3)],
]
mor, cow_twin = HudiTable("MERGE_ON_READ"), HudiTable("COPY_ON_WRITE")
m1, m2, m3 = (mor.upsert(batch) for batch in batches)
for batch in batches:
    cow_twin.upsert(batch)

fg1 = mor.file_groups["fg1"]
print("base versions:", [i for i, _ in fg1.base_versions], "log files:", [i for i, _ in fg1.log_files])
print([e["action"] for e in mor.timeline])

assert len(fg1.base_versions) == 1 and len(fg1.log_files) == 2
assert len(cow_twin.file_groups["fg1"].base_versions) == 3
statuses = lambda table: {r["order_id"]: r["status"] for r in table.snapshot()}
assert statuses(mor) == statuses(cow_twin)  # same logical result, different physical cost

Interview trap: MOR with a large compaction backlog looks fast to write and terrible to read. Always pair MOR with an explicit compaction SLA and alerting on delta-commit lag.

Common mistake: switching a table type in place. `hoodie.datasource.write.table.type` is fixed when the table is created. Converting COW to MOR, or the reverse, needs a rewrite or a migration job, not an option change on the next write.

## How do you choose between COW and MOR?

Choose COW when read latency and query simplicity are most important and updates are moderate. Choose MOR when write throughput and near-real-time updates are critical, and you can manage compaction and read-time merges. The choice also depends on engine support and operational constraints.

+ If updates touch a small fraction of rows and reads dominate, COW is usually cheaper end to end.
+ If ingestion is a continuous stream and a 5-15 minute freshness SLA matters, MOR ingests faster and compacts on a schedule.
+ If downstream engines cannot read Hudi log files, COW avoids a whole class of compatibility problems.
+ If write amplification is the bottleneck (network or object-store PUT limits), MOR is the escape valve.
+ If your team cannot operate a compaction job reliably, prefer COW; an unmanaged MOR table degrades continuously.

## What are the Hudi query types and how do you run an incremental pull?

Hudi exposes three query types from Spark, selected with `hoodie.datasource.query.type`.

+ `snapshot` (default): the latest committed view. On MOR it merges base files with log files at read time.
+ `read_optimized`: base files only. On COW it is identical to snapshot; on MOR it returns the last compacted state and can be stale.
+ `incremental`: only records committed in a given instant range, which powers incremental pipelines and CDC sinks.

In [ ]:
snapshot = {r["order_id"]: r["status"] for r in mor.snapshot()}
read_optimized = {r["order_id"]: r["status"] for r in mor.read_optimized()}
print("snapshot:      ", snapshot)
print("read_optimized:", read_optimized)
assert snapshot[1001] == "SHIPPED" and read_optimized[1001] == "CREATED"  # RO skips logs, so it is stale

# Incremental pull with a persisted checkpoint (the last instant processed)
checkpoint = m1
changed = mor.incremental(checkpoint)
print("changed since", checkpoint, "->", sorted(r["order_id"] for r in changed))
checkpoint = mor.completed()[-1]  # save after the downstream write succeeds
assert sorted(r["order_id"] for r in changed) == [1001, 1002, 1004]
assert mor.incremental(checkpoint) == []

In Spark:

```python
changes = (
    spark.read.format("hudi")
    .option("hoodie.datasource.query.type", "incremental")
    .option("hoodie.datasource.read.begin.instanttime", last_checkpoint)
    .load("s3://lake/hudi/orders")
)
```

## What is compaction in Hudi?

Compaction merges delta log files into base columnar files (mostly relevant for MOR tables). It improves read performance and reduces the overhead of reading many log files. Compaction must be scheduled and monitored to avoid backlog.

Compaction has two phases: a **planning** phase that writes a `compaction.requested` instant, and an **execution** phase that writes new base files and commits a `commit` that retires the log files.

In [ ]:
compaction = mor.compact()
fg1 = mor.file_groups["fg1"]
print("after compaction: base versions", [i for i, _ in fg1.base_versions], "logs", fg1.log_files)
assert not fg1.log_files
assert mor.read_optimized() == mor.snapshot()  # read_optimized is fresh right after compaction
assert mor.timeline[-1]["action"] == "compaction"

```sql
-- Async: schedule, then run, from a separate job
CALL run_compaction(op => 'schedule', table => 'lake.orders_hudi');
CALL run_compaction(op => 'run', table => 'lake.orders_hudi');
CALL show_compaction(table => 'lake.orders_hudi');
```

## What is clustering in Hudi and when do you need it?

Clustering reorganizes data files to improve layout (file sizes, sorting, locality) and query performance. You use clustering to combat small files, improve pruning, or optimize for common access patterns (for example sorting by event time).

Clustering rewrites selected file groups into better-shaped ones and commits the swap atomically as a `replacecommit`, so readers never see a partial state.

```python
clustering_options = {
    "hoodie.clustering.inline": "false",
    "hoodie.clustering.async.enabled": "true",
    "hoodie.clustering.plan.strategy.small.file.limit": "629145600",
    "hoodie.clustering.plan.strategy.target.file.max.bytes": "1073741824",
    "hoodie.clustering.plan.strategy.sort.columns": "customer_id,order_ts",
}
spark.sql("CALL run_clustering(table => 'lake.orders_hudi', op => 'scheduleAndExecute')")
```

+ After many small streaming commits, to consolidate hundreds of tiny files.
+ To sort data by a high-selectivity column so Parquet min/max statistics prune more row groups.
+ To change file sizing as data volume grows, without rewriting the whole table.
+ To fix skewed partitions, though a better partition key is usually the real fix.

## Why does the small files problem happen and how do you mitigate it in Hudi?

It happens when writers commit frequently with small batches or when partitions are highly granular. Mitigations include tuning write parallelism and file sizing, using clustering/compaction, and adjusting partitioning strategy to reduce tiny partitions.

1. Write fewer, larger commits: a 5-minute micro-batch of 500 MB is far healthier than a 1-minute batch of 100 MB.
2. Set `hoodie.parquet.small.file.limit` so Hudi merges new records into existing small files instead of creating new ones.
3. Right-size shuffle parallelism: target roughly `data_size / 128 MB` tasks, not a fixed 200.
4. Coarsen partitioning. Daily partitions with 2-5 GB each are usually better than hourly partitions with 100 MB each.
5. Run clustering to consolidate existing files and sort them for pruning.
6. Schedule cleaning so old file versions do not inflate file counts and storage.

## What table services does Hudi run and how do you schedule them?

Hudi table services are background operations that keep the table healthy. They can run inline inside a write or asynchronously in separate jobs.

| Service | Purpose | Key config |
|---|---|---|
| Compaction | Fold MOR log files into base files | `hoodie.compact.inline`, `hoodie.compact.inline.max.delta.commits` |
| Clustering | Reorganize file sizes and sort order | `hoodie.clustering.inline`, `hoodie.clustering.plan.strategy.sort.columns` |
| Cleaning | Delete old file versions and log files | `hoodie.clean.automatic`, `hoodie.cleaner.policy`, `hoodie.cleaner.commits.retained` |
| Archival | Move old timeline instants out of the active timeline | `hoodie.archivelog.folder`, `hoodie.keep.min.commits` |
| Indexing | Maintain the metadata table and stats index | `hoodie.metadata.enable`, `hoodie.metadata.index.column.stats.enable` |
| Rollback | Remove files from failed instants | automatic on failure |
| Savepoint | Pin a version so cleaning cannot delete it | `call create_savepoint(...)` |

Cleaning keeps a bounded number of file versions; a query that needs an older version fails once it is cleaned:

In [ ]:
fg1 = cow.file_groups["fg1"]
print("fg1 versions before clean:", [i for i, _ in fg1.base_versions])
removed = cow.clean(versions_retained=2)
print("removed", removed, "versions; remaining:", [i for i, _ in fg1.base_versions])
assert len(fg1.base_versions) == 2
assert cow.snapshot()  # the latest snapshot is untouched

## How do you handle CDC with Hudi?

Hudi can ingest CDC events and apply them as upserts/deletes to maintain a current-state table. You typically standardize event ordering and deduplication (record key + precombine field) and monitor lag/compaction. Some setups also keep an append-only log for auditing alongside the current-state table.

+ **DeltaStreamer** reads Kafka or a source table, deduplicates by record key and precombine, and upserts into Hudi.
+ **Kafka Connect Hudi sink** writes connector output directly into a Hudi table.
+ **Flink Hudi connector** consumes a changelog stream and supports `changelog` mode for `INSERT`/`UPDATE_BEFORE`/`UPDATE_AFTER`/`DELETE`.
+ **Spark Structured Streaming** with `foreachBatch` calling the Hudi datasource per micro-batch.

+ Order by a source log sequence number (`source_lsn`, `binlog_pos`) rather than wall-clock time; clock skew across shards breaks precombine ordering.
+ Handle tombstones explicitly: a delete event must either use `operation=delete` or carry a null ordering value for soft deletion.
+ Deduplicate within a micro-batch before the upsert, otherwise Hudi resolves duplicates by precombine and you lose the ability to log them.
+ Keep an append-only bronze table of raw CDC events for replay and audit; the Hudi current-state table should be reproducible from it.
+ Watch end-to-end lag: Kafka consumer lag plus commit latency plus compaction backlog.

Tombstones. A delete event can be applied as `operation=delete`, which removes the record, or as an upsert with a null precombine value, which leaves a soft-deleted row. Pick one convention for the whole pipeline and document it, because readers must know whether a missing key means deleted or never seen.

## What are common operational metrics for Hudi tables?

+ **Commit duration and rate**: time from batch start to commit on the timeline; alert on p95 regression.
+ **Compaction lag**: number of delta commits since the last compaction, and pending compaction instants. This is the single most important MOR metric.
+ **Clean backlog**: number of file versions beyond retention, and archive timeline growth.
+ **File metrics**: file count per partition, average and p95 file size, small-file ratio, and file-group count.
+ **Write amplification**: bytes read versus bytes written per upsert batch; a ratio near 10x means the index or partition strategy is wrong.
+ **Failed and rolled-back instants**: any rollback deserves investigation.
+ **Metadata table health**: metadata table size and files index freshness.
+ **Read metrics**: scan bytes, snapshot versus read-optimized latency, and query planning time (which improves with the metadata table).

## When would you choose Hudi vs Iceberg vs Delta?

Hudi is often chosen for CDC-heavy and upsert/delete workloads with near-real-time ingestion requirements. Iceberg is often chosen for engine-agnostic open tables with strong metadata and evolution features. Delta is often chosen when you are aligned with the Delta ecosystem and its tooling. The best choice depends on engines, governance, and operational capabilities.

| Dimension | Hudi | Iceberg | Delta Lake |
|---|---|---|---|
| Primary strength | Upserts, incremental pull, table services | Open multi-engine standard, hidden partitioning | Delta/Databricks ecosystem maturity |
| Update model | Indexed upsert; COW or MOR | Merge-on-read deletes; copy-on-write rewrite | Copy-on-write plus deletion vectors |
| Incremental read | First-class incremental query | Incremental append via snapshots/changelog | Change Data Feed |
| Partition evolution | Clustering, limited spec change | Strong partition evolution | Limited; requires rewrite |
| Streaming ingestion | Strong (DeltaStreamer, Flink, Connect) | Good (Flink, Spark) | Strong within Databricks |
| Engine interoperability | Good, but logs need Hudi-aware readers | Broadest | Broad, Iceberg via UniForm |
| Operational burden | Table services (compaction/clustering) to manage | Maintenance (compaction, expire snapshots) | Maintenance/optimize jobs |

Common mistake: defending a single format as universally best. The honest answer ties the choice to update patterns, read freshness, engine mix, and who operates the table services. A hybrid Hudi-for-ingestion, Iceberg-or-Delta-for-serving layout is common and defensible.

## Real world use cases for Apache Hudi

### 1. E-commerce order events with late-arriving updates

Business context: a marketplace needs a near-real-time order table that reflects cancellations, refunds, and late-arriving status events. Volume: ~40M order events/day (~120 GB/day), peak 15k events/s, freshness target 5 minutes.

Why this design: MOR absorbs the streaming write rate without rewriting Parquet on every event, `source_lsn` makes out-of-order events deterministic, and read-optimized queries keep dashboards fast between compactions. The incremental pull feeds the silver layer without rescanning history.

### 2. Fintech ledger with soft deletes and audit

Business context: a payments ledger must reflect reversals and account closures while retaining an auditable record. Volume: ~8M ledger entries/day, ~2 KB each, strict correctness.

Why this design: COW gives the strongest read semantics for audit queries, the bronze CDC log makes the ledger reproducible, and soft deletes preserve history for regulators while `_hoodie_is_deleted`-style filtering hides reversed entries from operational reads. A separate hard-delete job handles GDPR erasure.

### 3. IoT telemetry current-state and incremental analytics

Business context: 200k devices emit readings every 30 seconds; operations needs the latest device state and analytics needs only changed rows. Volume: ~600M events/day (~90 GB/day).

Why this design: one Hudi table serves both the low-latency current-state read and the change feed, removing the need to duplicate data. DeltaStreamer handles Kafka offset commit and schema registry integration, and incremental pulls keep the silver aggregation job cheap.

### 4. Multi-tenant SaaS activity lake with tenant isolation

Business context: a B2B SaaS platform ingests customer activity into one governed lake and must serve per-tenant queries with cost attribution. Volume: 500 tenants, ~150M events/day, uneven tenant sizes.

Why this design: COW keeps tenant-facing queries simple and fast, composite keys prevent cross-tenant collisions, and date partitioning with column statistics lets Spark prune to a tenant's slices. Column stats in the metadata table keep planning fast despite the file count. Very large tenants can later be split into their own table to avoid skew.

### 5. CDC into a lakehouse bronze layer

Business context: an enterprise replicating 30 operational databases into one lakehouse wants a single upsert-capable bronze table per source before building conformed silver models. Volume: ~1.2B change events/day across sources.

## How do you tune Hudi tables for performance?

Tuning Hudi means balancing four things: write latency, read latency, storage, and operational cost. Almost every knob moves work between them. The order below is the order that pays off in practice: fix file sizing first, then the table type and compaction, then metadata and indexes, then layout.

### Size files before anything else

File size is the single biggest driver of read performance. Too many small files add object store requests, task scheduling overhead, and poor compression. Too few huge files hurt parallelism and make an upsert rewrite more than it needs to.

In [ ]:
# Sizing the write parallelism from the batch and the target file size.
TARGET_FILE_BYTES = 128 * 1024 * 1024


def write_parallelism(batch_bytes: int, target_bytes: int = TARGET_FILE_BYTES) -> int:
    """One task per target-sized chunk, with a floor of 1."""
    return max(1, round(batch_bytes / target_bytes))


print(write_parallelism(2 * 1024 ** 3))   # 2 GB batch -> 16 tasks

### Choose COW or MOR from the workload, not the fashion

COW pays on every write and saves on every read. MOR saves on writes and pays at read time until compaction runs. With a few percent of rows updated and many readers, COW is usually cheaper end to end. With continuous updates and few readers, MOR absorbs the write rate.

### Schedule compaction explicitly

Compaction is what keeps a MOR table readable. Schedule it by delta-commit count with `hoodie.compact.inline.max.delta.commits`, or by time with `hoodie.compact.inline.trigger.strategy=TIME_ELAPSED`. Inline compaction keeps operations simple but adds latency to the writer. Async compaction, which is the default in recent versions, runs beside ingestion and is the right choice once ingestion is continuous.

### Use the metadata table for file listing

On cloud object storage, listing a table with hundreds of thousands of files is slow and can hit request rate limits. The metadata table stores the files index, the column stats index, and other indexes as a Hudi table under `.hoodie/metadata`. Readers then resolve the file list from one lookup instead of a full listing.

### Pick the index from key cardinality and update volume

`hoodie.index.type` decides how an upsert finds the file groups that hold its keys. Bloom is the general-purpose default. HFile trades extra files for faster repeat lookups on large tables. Simple keeps the key map in driver memory, which is only safe when the key count is small. Bucket removes the lookup entirely but fixes the layout at creation time.

### Cluster for layout, not as a first fix

Clustering rewrites groups of files into larger, optionally sorted files and commits the swap as a `replacecommit`. It is the right tool for consolidating files after a period of small commits and for sorting on a column that queries filter on.

### Design the partition path to match queries

A good partition path is a low-cardinality column that queries filter on and that keeps each partition at a healthy size, usually 1 to 5 GB. Daily partitions are the common default. Partitioning by a high-cardinality id, or by hour when volume is low, creates thousands of tiny partitions and undoes every file-sizing improvement.

### Tune cleaning and retention against query latency

Cleaning deletes file versions that are older than the retention window. Too little retention and a long-running query or an incremental consumer fails on a missing version. Too much retention and storage grows without bound.

## Hudi exercises

These exercises use only the standard library. The numbers are illustrative; the point is knowing which quantity to compute.

### Try it 1. Choose COW or MOR from a read/write profile

**Task.** Write a function `pick_table_type` that returns `"MERGE_ON_READ"` when the freshness target is 15 minutes or less and there are fewer than 5 readers, otherwise `"COPY_ON_WRITE"`. Apply it to a streaming profile (5-minute freshness, 1 reader) and a BI profile (60-minute freshness, 40 readers).

**Hint.** `freshness_minutes <= 15 and readers < 5` is the MOR condition.

**Expected result.** `MERGE_ON_READ` for the stream, `COPY_ON_WRITE` for BI.

In [ ]:
def pick_table_type(freshness_minutes: int, readers: int) -> str:
    if freshness_minutes <= 15 and readers < 5:
        return "MERGE_ON_READ"
    return "COPY_ON_WRITE"


stream = pick_table_type(freshness_minutes=5, readers=1)
bi = pick_table_type(freshness_minutes=60, readers=40)

print("streaming profile:", stream)
print("BI profile:", bi)

assert stream == "MERGE_ON_READ"
assert bi == "COPY_ON_WRITE"

### Try it 2. Size files from ingestion rate and target file size

**Task.** A stream ingests 120 GB per day in five-minute micro-batches. Compute the bytes per batch and the number of 128 MiB target files the writer should aim for. Use 1 GB = 1024 MiB and `math.ceil`.

**Hint.** There are 24 * 60 / 5 = 288 five-minute batches per day. Files per batch is `ceil(batch_mib / 128)`.

**Expected result.** About 426.7 MiB per batch and 4 files per batch.

In [ ]:
import math

TARGET_MIB = 128
daily_mib = 120 * 1024
batches_per_day = (24 * 60) // 5
batch_mib = daily_mib / batches_per_day
files_per_batch = math.ceil(batch_mib / TARGET_MIB)

print(f"batches per day = {batches_per_day}")
print(f"batch size = {batch_mib:.1f} MiB")
print(f"target files per batch = {files_per_batch}")

assert batches_per_day == 288
assert round(batch_mib, 1) == 426.7
assert files_per_batch == 4

### Try it 3. Compute the compaction trigger from delta-commit count

**Task.** A MOR table ran 4 delta commits after its last completed compaction. With `hoodie.compact.inline.max.delta.commits=5`, the trigger fires when pending delta commits reach 5. Return whether compaction should be scheduled now, and the number of commits still needed.

**Hint.** `should_compact = pending >= threshold` and `remaining = max(0, threshold - pending)`.

**Expected result.** Not yet, 1 commit remaining.

In [ ]:
threshold = 5
pending = 4

should_compact = pending >= threshold
remaining = max(0, threshold - pending)

print("schedule compaction now:", should_compact)
print("delta commits remaining:", remaining)

assert should_compact is False
assert remaining == 1
assert (4 + remaining) >= threshold

### Try it 4. Estimate the metadata-table benefit from file counts

**Task.** A reader lists 200,000 files directly, at 2 ms per list request, and compares that with the metadata table, which resolves the same listing in 50 ms. Compute both times, the speed-up factor, and whether the metadata table saves more than 5 minutes of planning time.

**Hint.** `direct_seconds = file_count * ms_per_call / 1000`. The factor is `direct_seconds / metadata_seconds`.

**Expected result.** 400.0 s direct, 0.05 s with metadata, 8000x faster, planning saved.

In [ ]:
file_count = 200_000
ms_per_list_call = 2
metadata_seconds = 0.05

direct_seconds = file_count * ms_per_list_call / 1000
speedup = direct_seconds / metadata_seconds
saves_minutes = (direct_seconds - metadata_seconds) > 5 * 60

print(f"direct listing = {direct_seconds:.1f} s")
print(f"with metadata table = {metadata_seconds:.2f} s")
print(f"speed-up = {speedup:.0f}x")
print("saves over 5 minutes:", saves_minutes)

assert direct_seconds == 400.0
assert speedup == 8000.0
assert saves_minutes is True

### Try it 5. Pick an index type from key cardinality

**Task.** Write `pick_index(table_keys, partitioned, streaming, keys_move_partitions)` that returns the Hudi index type. Rules: a global index when keys move partitions, `BUCKET` for streaming with a stable key space, `SIMPLE` when the table has at most 5 million keys, `HFILE` for a large partitioned table, and `BLOOM` otherwise.

**Hint.** Check the rules in priority order and return the first match.

**Expected result.** `GLOBAL_BLOOM`, `BUCKET`, `SIMPLE`, `HFILE`, `BLOOM`.

In [ ]:
def pick_index(
    table_keys: int,
    partitioned: bool,
    streaming: bool,
    keys_move_partitions: bool,
) -> str:
    if keys_move_partitions:
        return "GLOBAL_BLOOM"
    if streaming:
        return "BUCKET"
    if table_keys <= 5_000_000:
        return "SIMPLE"
    if partitioned:
        return "HFILE"
    return "BLOOM"


cases = [
    (pick_index(50_000_000, True, True, True), "GLOBAL_BLOOM"),
    (pick_index(50_000_000, True, True, False), "BUCKET"),
    (pick_index(1_000_000, False, False, False), "SIMPLE"),
    (pick_index(500_000_000, True, False, False), "HFILE"),
    (pick_index(500_000_000, False, False, False), "BLOOM"),
]

for got, _ in cases:
    print(got)

for got, expected in cases:
    assert got == expected

### Try it 6. Plan cleaning retention against query latency

**Task.** The longest query against a table takes 5 hours. Commits happen every 30 minutes. Compute the minimum number of commits to retain under `KEEP_LATEST_COMMITS` so the oldest file version survives the longest reader, then add a 20% safety margin and round up.

**Hint.** `minimum = ceil(longest_query_minutes / commit_interval_minutes)`, then `retained = ceil(minimum * 1.2)`.

**Expected result.** 10 commits minimum, 12 with margin.

In [ ]:
import math

longest_query_minutes = 5 * 60
commit_interval_minutes = 30

minimum = math.ceil(longest_query_minutes / commit_interval_minutes)
retained = math.ceil(minimum * 1.2)

print("minimum commits to retain:", minimum)
print("with 20% margin:", retained)

assert minimum == 10
assert retained == 12
assert retained >= minimum

### Try it 7. Estimate file count after clustering

**Task.** A partition holds 8,000 small files totalling 480 GiB. Clustering targets 1 GiB files. Estimate the file count after clustering, the reduction factor, and check that the new count is at least 1.

**Hint.** `after = max(1, math.ceil(total_gib / target_gib))`. The factor is `before / after`.

**Expected result.** 480 files, 16.7x fewer.

In [ ]:
import math

files_before = 8_000
total_gib = 480
target_gib = 1

files_after = max(1, math.ceil(total_gib / target_gib))
reduction = files_before / files_after

print("files before:", files_before)
print("files after:", files_after)
print(f"reduction = {reduction:.1f}x")

assert files_after == 480
assert round(reduction, 1) == 16.7
assert files_after >= 1